#TODO

- prova multipli nn su 27 modelli
- function documentation
- import all the functions
- hyperparam refined
- train full df
- predictions



# Project

In [3]:
import pandas as pd
#from thefuzz import process
#from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# For imputation
#from sklearn.experimental import enable_iterative_imputer  # noqa
#from sklearn.impute import IterativeImputer

from scipy import stats
from sklearn.feature_selection import VarianceThreshold, RFE
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier #, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor

import numpy as np
from sklearn.base import BaseEstimator, clone
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from typing import Dict

from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.neighbors import KNeighborsRegressor

from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor

import random
from copy import deepcopy
import time

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.cluster import DBSCAN

In [ ]:
import sys
import os

# Add the current directory to sys.path so we can import from Functions
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

# Import Preprocessing functions
from Functions.preprocessing import (
    simple_processing, 
    remove_price_outliers, 
    fit_transform_encoding, 
    decode, 
    minimal_features
)

# Import Imputation functions
from Functions.imputation import (
    mode_imputation, 
    fit_imputer, 
    apply_imputer, 
    fit_price_imputer, 
    apply_price_imputer
)

In [6]:
train_df = pd.read_csv("https://raw.githubusercontent.com/LiberoBiagi/ML_Nova_IMS_25-26/refs/heads/main/data/train.csv")

In [7]:
train_df = train_df.set_index("carID")
train_df = train_df.drop(columns=["paintQuality%"])

In [9]:
#train_df.head()

### Making sure that the model prices are not typos

In [ ]:
"""
#df_cleaned = simple_processing(train_df)
#df_cleaned = df_cleaned[df_cleaned["model"].notna()]
#df_price_model = df_cleaned[["Brand","model", "price"]].copy()

# Get the list of unique models
models = df_price_model['model'].unique()

# Loop through each model and create a boxplot
#for model in models:
sns.catplot(data=df_price_model, y='price', col='model', kind='box', col_wrap=4, height=3, sharey=False)
plt.show()"""

"# Get the list of unique models\nmodels = df_price_model['model'].unique()\n\n# Loop through each model and create a boxplot\n#for model in models:\nsns.catplot(data=df_price_model, y='price', col='model', kind='box', col_wrap=4, height=3, sharey=False)\nplt.show()"

## Data cleaning

## Workflow for Train and Validation Sets

TODO: Explanation

#### Encoding

In [10]:
# Fix typos and small numeric cleanup
df = simple_processing(train_df)

# Remove extreme price outliers
df = remove_price_outliers(df)

# Encode cateogrical columns and replace the str with int columns. Return fitted encoders for decoding at the end
df_encoded, encoders = fit_transform_encoding(df)

#### Creating the stratification column

In [11]:
# Create Categorical price column with 0 < 1 < 2 for the price
df_encoded["price_cat"] = pd.qcut(df_encoded["price"], 3, labels=False)

# Combine the 10 unique brand values (1-9 & NA) with the 3 unique price category values (0-2)
stratify_col = df_encoded["Brand_transformed"].astype(str) + "_" + df_encoded["price_cat"].astype(str)

#### Imputation and decoding

In [12]:
# Train validation split
train_split, validation_split = train_test_split(df_encoded, test_size=0.2, random_state=69, stratify=stratify_col)
# train_split, validation_split = train_test_split(df_encoded, test_size=0.2, random_state=42)

# Drop price_cat column from both splits as it would introduce data leakage
train_split = train_split.drop("price_cat", axis=1)
validation_split = validation_split.drop("price_cat", axis=1)

train_split = mode_imputation(train_split)
validation_split = mode_imputation(validation_split)

# Train imputer on train (data leakage risk thus only train on train_split)
imputer = fit_imputer(train_split)

# Apply trained imputer to both datasplits
imputed_train = apply_imputer(train_split, imputer)
imputed_validation = apply_imputer(validation_split, imputer)

# Impute the missing price values created by removing extrem outliers
# In our step to remove extrem price outliers we introduced missing values in the price column. However, in our normal imputer, which we also use for imputing the test 
# Dataset we can not fix these values as this would cause the imputer to not work on the test subset, with no price feature. 
# Thus we create a seperate price imputer to fix the missing values in the testing and validation sets. This imputer is trained on train and applied to validation
# Fit price imputer
price_imputer = fit_price_imputer(imputed_train)
# Apply price imputer
imputed_train = apply_price_imputer(imputed_train, price_imputer)
imputed_validation = apply_price_imputer(imputed_validation, price_imputer)

# Decode encoded columns using the fitted encoders
train_processed = decode(imputed_train, encoders)
validation_processed = decode(imputed_validation, encoders)

c:\Users\morit\anaconda3\envs\DataMining\Lib\site-packages\sklearn\impute\_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


## Workflow for Cleaning Testing Dataset

We use all the information available from the training data sets (train and validation splits) to fit imputers and preprocess the testing dataset. This should create a scenario as close to real world deployment as possible, where we use our existing data to process new data entries.

In [13]:
test_df = pd.read_csv("https://raw.githubusercontent.com/LiberoBiagi/ML_Nova_IMS_25-26/refs/heads/main/data/test.csv")

In [14]:
test_df = test_df.set_index("carID")
test_df = test_df.drop(columns=["paintQuality%"])

### Simple pre Imputation processing of Test data

In [20]:
test = simple_processing(test_df)

test_df_encoded, test_encoders = fit_transform_encoding(test)

### Fit the full testing imputer

In [18]:
# Preprocess the full training dataframe from scratch
df = simple_processing(train_df)
df = remove_price_outliers(df)
df_encoded, encoders = fit_transform_encoding(df)
train_df_mode_imputed = mode_imputation(df_encoded)

# Train a full imputer on the all available training data
full_training_imputer = fit_imputer(train_df_mode_imputed)

c:\Users\morit\anaconda3\envs\DataMining\Lib\site-packages\sklearn\impute\_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


### Apply the full imputer to the testing data

In [23]:
#df_encoded_no_price = test_df_imputed.drop(["price", "price_cat"], axis=1)

# Apply imputer trained on testing data
test_df_imputed = apply_imputer(test_df_encoded, full_training_imputer)

test_processed = decode(test_df_imputed, test_encoders)

# MARCEL PART

In [24]:
train_val_models = set(pd.concat([train_processed, validation_processed])["model"].unique())
test_models = set(test_processed["model"].unique())

missing_models = train_val_models - test_models


In [25]:
list(missing_models)


['s8',
 'streetka',
 'getz',
 'ranger',
 '200',
 'accent',
 'caddy maxi',
 '230',
 'escort',
 'urban cruiser',
 'verso-s',
 '220',
 'kadjar',
 'a2']

## Feature Engineering

In [26]:
train_processed.index = train_processed.index.astype(int)
validation_processed.index = validation_processed.index.astype(int)

In [27]:
X_train = train_processed
X_val = validation_processed

In [28]:
def minimal_features(df):
    df = df.copy()
    df['age'] = 2020 - df['year']
    df = df.drop(columns=["year"])
    df['mileage_per_year'] = df['mileage'] / (df['age'] + 1)
    df['efficiency_ratio'] = df['mpg'] / (df['engineSize'] + 0.1)
    df['age_mileage'] = df['age'] * df['mileage'] / 100000

    return df



In [29]:
X_train = minimal_features(X_train)
X_val = minimal_features(X_val)

In [30]:
model_mean_price = X_train.groupby('model')['price'].mean() 
X_train['model_mean_price'] = X_train['model'].map(model_mean_price) 
X_val['model_mean_price'] = X_val['model'].map(model_mean_price)

In [31]:
X_train = X_train.drop(columns=["model"])
X_val = X_val.drop(columns=["model"])

In [32]:
y_train = X_train['price']
y_val = X_val['price']

In [33]:
X_train = X_train.drop(columns=['price'])

X_val = X_val.drop(columns=['price'])


In [34]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60778 entries, 0 to 60777
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   mileage           60778 non-null  int64  
 1   tax               60778 non-null  int64  
 2   mpg               60778 non-null  float64
 3   engineSize        60778 non-null  float64
 4   previousOwners    60778 non-null  int64  
 5   stated_no_damage  60778 non-null  float64
 6   Brand             60778 non-null  object 
 7   transmission      60778 non-null  object 
 8   fuelType          60778 non-null  object 
 9   age               60778 non-null  int64  
 10  mileage_per_year  60778 non-null  float64
 11  efficiency_ratio  60778 non-null  float64
 12  age_mileage       60778 non-null  float64
 13  model_mean_price  60778 non-null  float64
dtypes: float64(7), int64(4), object(3)
memory usage: 6.5+ MB


In [35]:
X_val.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15195 entries, 0 to 15194
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   mileage           15195 non-null  int64  
 1   tax               15195 non-null  int64  
 2   mpg               15195 non-null  float64
 3   engineSize        15195 non-null  float64
 4   previousOwners    15195 non-null  int64  
 5   stated_no_damage  15195 non-null  float64
 6   Brand             15195 non-null  object 
 7   transmission      15195 non-null  object 
 8   fuelType          15195 non-null  object 
 9   age               15195 non-null  int64  
 10  mileage_per_year  15195 non-null  float64
 11  efficiency_ratio  15195 non-null  float64
 12  age_mileage       15195 non-null  float64
 13  model_mean_price  15195 non-null  float64
dtypes: float64(7), int64(4), object(3)
memory usage: 1.6+ MB


In [36]:
y_train.info()

<class 'pandas.core.series.Series'>
RangeIndex: 60778 entries, 0 to 60777
Series name: price
Non-Null Count  Dtype  
--------------  -----  
60778 non-null  float64
dtypes: float64(1)
memory usage: 475.0 KB


In [37]:
y_val.info()

<class 'pandas.core.series.Series'>
RangeIndex: 15195 entries, 0 to 15194
Series name: price
Non-Null Count  Dtype  
--------------  -----  
15195 non-null  float64
dtypes: float64(1)
memory usage: 118.8 KB


## Test of the models on the full dataset

In [38]:
class BrandModelTrainer:
    def __init__(self, estimator):
        self.estimator = estimator
        self.brand_models = {}
        self.feature_cols = None

    def fit(self, X_train, y_train):
        self.feature_cols = [c for c in X_train.columns if c != "Brand"]
        print(f"Training models for {len(X_train['Brand'].unique())} brands...\n")

        for brand in X_train["Brand"].unique():
            mask = X_train["Brand"] == brand
            Xb = X_train.loc[mask, self.feature_cols]
            yb = y_train[mask]

            numeric_cols = Xb.select_dtypes(include=["int64", "int32" ,"float64"]).columns.tolist()
            categorical_cols = Xb.select_dtypes(include=["object", "category"]).columns.tolist()

            preprocessor = ColumnTransformer([
                ("num", RobustScaler(), numeric_cols),
                ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
            ])

            model = Pipeline([
                ("preprocess", preprocessor),
                ("estimator", clone(self.estimator))
            ])

            model.fit(Xb, yb)
            self.brand_models[brand] = model
            print(f"  ✓ {brand} done.")

        return self

    def predict(self, X):
        preds = np.zeros(len(X))
        for brand, model in self.brand_models.items():
            mask = X["Brand"] == brand
            if mask.sum() == 0:
                continue
            Xb = X.loc[mask, self.feature_cols]
            preds[mask] = model.predict(Xb)
        return preds

    def evaluate_train(self, X_train, y_train):
        y_pred = self.predict(X_train)
        rmse = np.sqrt(mean_squared_error(y_train, y_pred))
        mae = mean_absolute_error(y_train, y_pred)
        r2 = r2_score(y_train, y_pred)
        print("\nTraining Set Performance (Overall):")
        print(f"  RMSE: {rmse:.2f}")
        print(f"  MAE:  {mae:.2f}")
        print(f"  R²:   {r2:.4f}")
        return {"RMSE": rmse, "MAE": mae, "R²": r2}

    def evaluate(self, X_val, y_val):
        y_pred = self.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        mae = mean_absolute_error(y_val, y_pred)
        r2 = r2_score(y_val, y_pred)
        print("\nValidation Set Performance (Overall):")
        print(f"  RMSE: {rmse:.2f}")
        print(f"  MAE:  {mae:.2f}")
        print(f"  R²:   {r2:.4f}")
        return {"RMSE": rmse, "MAE": mae, "R²": r2}

    def evaluate_by_brand(self, X, y, split_name="Validation"):
        y_pred = self.predict(X)
        results = []
        for brand in X["Brand"].unique():
            mask = X["Brand"] == brand
            y_true_b = y[mask]
            y_pred_b = y_pred[mask]
            rmse = np.sqrt(mean_squared_error(y_true_b, y_pred_b))
            mae = mean_absolute_error(y_true_b, y_pred_b)
            r2 = r2_score(y_true_b, y_pred_b)
            results.append({"Brand": brand, "N": len(y_true_b), "RMSE": rmse, "MAE": mae, "R²": r2})
        df = pd.DataFrame(results).sort_values("RMSE")
        print(f"\n{split_name} Performance per Brand:")
        print(df.to_string(index=False))
        return df

    def evaluate_train_by_brand(self, X_train, y_train):
        return self.evaluate_by_brand(X_train, y_train, split_name="Training")
    
    def save_predictions(self, X, path):
        preds = self.predict(X)
        df = pd.DataFrame({
            "CarID": X["CarID"].values,
            "price": preds
        })
        df.to_csv(path, index=False)
        print(f"File salvato in: {path}")
        return df


In [41]:
#raise SystemExit("Stop before training the models")

### Linear Regression

In [42]:
LR = LinearRegression()
trainer = BrandModelTrainer(LR)

# fit
trainer.fit(X_train, y_train)

# performance overall
trainer.evaluate_train(X_train, y_train)
trainer.evaluate(X_val, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train, y_train)
trainer.evaluate_by_brand(X_val, y_val)

Training models for 9 brands...

  ✓ opel done.
  ✓ audi done.
  ✓ mercedes done.
  ✓ ford done.
  ✓ bmw done.
  ✓ toyota done.
  ✓ hyundai done.
  ✓ vw done.
  ✓ skoda done.

Training Set Performance (Overall):
  RMSE: 3451.10
  MAE:  2146.54
  R²:   0.8706

Validation Set Performance (Overall):
  RMSE: 3378.60
  MAE:  2146.61
  R²:   0.8741

Training Performance per Brand:
   Brand     N        RMSE         MAE       R²
    opel  7640 1820.940267 1258.038406 0.739457
    ford 13110 2115.475577 1486.041527 0.802222
   skoda  3504 2160.162219 1501.938959 0.875942
 hyundai  2724 2423.738071 1651.687723 0.838554
  toyota  3776 2424.533705 1488.493680 0.846946
      vw  8479 3164.500884 2106.312660 0.831688
    audi  5971 4468.931440 2912.149730 0.853660
     bmw  6037 4555.101513 3095.915974 0.839049
mercedes  9537 5179.988218 3360.414584 0.760750

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1907 1893.012996 1299.110166 0.717565
   skoda  881

,Brand,N,RMSE,MAE,R²
7,opel,1907,1893.012996,1299.110166,0.717565
8,skoda,881,2240.480606,1535.947815,0.868385
5,hyundai,687,2243.184900,1621.896771,0.853502
1,ford,3278,2246.715780,1490.218126,0.777431
3,toyota,940,2768.807529,1582.394431,0.830498
0,vw,2125,3083.038407,2135.915540,0.842650
6,audi,1490,4272.881130,2859.099780,0.857222
2,bmw,1505,4456.642789,3027.288735,0.845313
4,mercedes,2382,4921.308969,3335.661848,0.778288


### Decision Tree

In [43]:
Tree = DecisionTreeRegressor(criterion="absolute_error")
trainer = BrandModelTrainer(Tree)

# fit
trainer.fit(X_train, y_train)

# performance overall
trainer.evaluate_train(X_train, y_train)
trainer.evaluate(X_val, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train, y_train)
trainer.evaluate_by_brand(X_val, y_val)        # Performance VALID

Training models for 9 brands...

  ✓ opel done.
  ✓ audi done.
  ✓ mercedes done.
  ✓ ford done.
  ✓ bmw done.
  ✓ toyota done.
  ✓ hyundai done.
  ✓ vw done.
  ✓ skoda done.

Training Set Performance (Overall):
  RMSE: 104.41
  MAE:  5.77
  R²:   0.9999

Validation Set Performance (Overall):
  RMSE: 2717.14
  MAE:  1627.71
  R²:   0.9186

Training Performance per Brand:
   Brand     N       RMSE       MAE       R²
  toyota  3776  36.498308  1.571769 0.999965
    ford 13110  59.143592  3.164760 0.999845
      vw  8479  71.144767  4.987027 0.999915
   skoda  3504  81.232575  3.905251 0.999825
    audi  5971  91.272020  4.119578 0.999939
    opel  7640 115.867908 10.034293 0.998945
 hyundai  2724 122.123981  7.596549 0.999590
     bmw  6037 124.888708  3.789631 0.999879
mercedes  9537 164.661451 10.760931 0.999758

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1907 1534.792612 1030.244288 0.814343
 hyundai  687 1732.147224 1182.071665 0.912648


,Brand,N,RMSE,MAE,R²
7,opel,1907,1534.792612,1030.244288,0.814343
5,hyundai,687,1732.147224,1182.071665,0.912648
3,toyota,940,1869.821695,1186.005952,0.922698
1,ford,3278,1882.128912,1152.855623,0.843805
8,skoda,881,2021.426015,1365.042160,0.892863
0,vw,2125,2552.563894,1607.347802,0.892140
6,audi,1490,3481.589306,2234.856538,0.905207
4,mercedes,2382,3679.842512,2356.913161,0.876039
2,bmw,1505,3906.644091,2325.608900,0.881137


### KNR

In [44]:
KNR = KNeighborsRegressor()
trainer = BrandModelTrainer(KNR)

# fit
trainer.fit(X_train, y_train)

# performance overall
trainer.evaluate_train(X_train, y_train)
trainer.evaluate(X_val, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train, y_train)
trainer.evaluate_by_brand(X_val, y_val)

Training models for 9 brands...

  ✓ opel done.
  ✓ audi done.
  ✓ mercedes done.
  ✓ ford done.
  ✓ bmw done.
  ✓ toyota done.
  ✓ hyundai done.
  ✓ vw done.
  ✓ skoda done.

Training Set Performance (Overall):
  RMSE: 2087.10
  MAE:  1278.17
  R²:   0.9527

Validation Set Performance (Overall):
  RMSE: 2602.17
  MAE:  1607.67
  R²:   0.9253

Training Performance per Brand:
   Brand     N        RMSE         MAE       R²
    opel  7640 1053.369568  709.972862 0.912814
    ford 13110 1257.785857  861.041191 0.930084
  toyota  3776 1356.807409  860.440746 0.952068
 hyundai  2724 1581.592342 1002.412886 0.931254
   skoda  3504 1586.397607 1136.117614 0.933092
      vw  8479 1942.281431 1310.870678 0.936594
mercedes  9537 2831.116625 1838.563952 0.928532
    audi  5971 2906.105145 1816.370438 0.938116
     bmw  6037 2993.330574 1907.752829 0.930497

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1907 1413.309166  945.040416 0.842571
    ford 3278

,Brand,N,RMSE,MAE,R²
7,opel,1907,1413.309166,945.040416,0.842571
1,ford,3278,1654.822606,1062.651280,0.879254
5,hyundai,687,1681.360294,1156.872522,0.917696
8,skoda,881,1823.152122,1344.157726,0.912850
3,toyota,940,1991.106210,1135.496156,0.912345
0,vw,2125,2347.034894,1639.854301,0.908810
6,audi,1490,3325.835116,2230.596571,0.913499
2,bmw,1505,3602.446882,2343.213821,0.898927
4,mercedes,2382,3766.317872,2418.907461,0.870144


### Random Forest

In [45]:
RF = RandomForestRegressor(random_state=69)
rf_trainer = BrandModelTrainer(RF)


rf_trainer.fit(X_train, y_train)


rf_trainer.evaluate_train(X_train, y_train)
rf_trainer.evaluate(X_val, y_val)

rf_trainer.evaluate_train_by_brand(X_train, y_train)
rf_trainer.evaluate_by_brand(X_val, y_val)



Training models for 9 brands...

  ✓ opel done.
  ✓ audi done.
  ✓ mercedes done.
  ✓ ford done.
  ✓ bmw done.
  ✓ toyota done.
  ✓ hyundai done.
  ✓ vw done.
  ✓ skoda done.

Training Set Performance (Overall):
  RMSE: 778.75
  MAE:  468.95
  R²:   0.9934

Validation Set Performance (Overall):
  RMSE: 2027.75
  MAE:  1254.55
  R²:   0.9547

Training Performance per Brand:
   Brand     N        RMSE        MAE       R²
    opel  7640  467.466688 300.273072 0.982829
    ford 13110  507.914036 343.069032 0.988599
  toyota  3776  548.127567 334.022290 0.992177
 hyundai  2724  610.205448 371.660006 0.989767
   skoda  3504  624.784356 410.416263 0.989622
      vw  8479  716.999618 455.181956 0.991359
    audi  5971 1010.132487 628.499576 0.992523
     bmw  6037 1051.525778 641.963074 0.991423
mercedes  9537 1081.936465 682.631900 0.989562

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1907 1261.061787  826.756821 0.874662
 hyundai  687 1265.098545

,Brand,N,RMSE,MAE,R²
7,opel,1907,1261.061787,826.756821,0.874662
5,hyundai,687,1265.098545,892.610998,0.953404
3,toyota,940,1393.624389,927.351962,0.957058
8,skoda,881,1434.187029,1015.580090,0.946070
1,ford,3278,1438.352141,915.406146,0.908778
0,vw,2125,1864.178800,1221.533862,0.942471
6,audi,1490,2562.367951,1682.489444,0.948654
2,bmw,1505,2606.440047,1674.686016,0.947090
4,mercedes,2382,2936.111831,1881.960610,0.921083


### Neural Network

In [46]:
from sklearn.neural_network import MLPRegressor

mlp_deep = MLPRegressor(
    hidden_layer_sizes=(256, 128, 64),  
    activation='relu',
    solver='adam',
    alpha=0.0001,  
    learning_rate_init=0.001,
    learning_rate='adaptive',
    max_iter=2000,
    batch_size=256,
    random_state=69,
    early_stopping=True,
    n_iter_no_change=30,  
    validation_fraction=0.15,
    verbose=True
)

mlp_trainer = BrandModelTrainer(mlp_deep)


mlp_trainer.fit(X_train, y_train)


mlp_trainer.evaluate_train(X_train, y_train)
mlp_trainer.evaluate(X_val, y_val)


mlp_trainer.evaluate_train_by_brand(X_train, y_train)
mlp_trainer.evaluate_by_brand(X_val, y_val)

Training models for 9 brands...

Iteration 1, loss = 60111864.13044062
Validation score: -7.709747
Iteration 2, loss = 59282110.35702281
Validation score: -7.419917
Iteration 3, loss = 54952942.34602675
Validation score: -6.231930
Iteration 4, loss = 42148467.32360731
Validation score: -3.543174
Iteration 5, loss = 21787892.62978505
Validation score: -0.725817
Iteration 6, loss = 8364469.90237840
Validation score: 0.275050
Iteration 7, loss = 5086909.91066059
Validation score: 0.428377
Iteration 8, loss = 4129072.72753506
Validation score: 0.501465
Iteration 9, loss = 3456362.85398503
Validation score: 0.565016
Iteration 10, loss = 3055046.94756908
Validation score: 0.602447
Iteration 11, loss = 2754357.85856672
Validation score: 0.631723
Iteration 12, loss = 2493046.44782003
Validation score: 0.654691
Iteration 13, loss = 2297959.17005230
Validation score: 0.672515
Iteration 14, loss = 2142533.92213469
Validation score: 0.684756
Iteration 15, loss = 2023008.35796549
Validation score: 

,Brand,N,RMSE,MAE,R²
7,opel,1907,1294.647665,872.751475,0.867896
5,hyundai,687,1512.684519,1070.760884,0.933381
1,ford,3278,1571.041086,1023.167182,0.891171
8,skoda,881,1680.440120,1195.354945,0.925960
3,toyota,940,1775.131177,1048.794443,0.930329
0,vw,2125,2241.242017,1485.990807,0.916846
6,audi,1490,2987.781788,2023.078248,0.930190
2,bmw,1505,3047.281032,2044.076017,0.927679
4,mercedes,2382,3428.452434,2274.602822,0.892397


### Performance

| *Model* | *RMSE* | *MAE* | *R²* |
|---------|--------|--------|------|
| Linear Regression | 3383.27 | 2156.67 | 0.8723 |
| Tree | 2784.00 | 1663.58 | 0.9135 |
| KNR|2623.89 |1599.60 |0.9232 |
|RF | 2032.89 |1258.34 |0.9539 |
|NN | 2319.24 |1466.96 |0.9400 |


 

## Feature selection - Filter Methods

In [47]:
df = X_train.join(y_train)

In [48]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60778 entries, 0 to 60777
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   mileage           60778 non-null  int64  
 1   tax               60778 non-null  int64  
 2   mpg               60778 non-null  float64
 3   engineSize        60778 non-null  float64
 4   previousOwners    60778 non-null  int64  
 5   stated_no_damage  60778 non-null  float64
 6   Brand             60778 non-null  object 
 7   transmission      60778 non-null  object 
 8   fuelType          60778 non-null  object 
 9   age               60778 non-null  int64  
 10  mileage_per_year  60778 non-null  float64
 11  efficiency_ratio  60778 non-null  float64
 12  age_mileage       60778 non-null  float64
 13  model_mean_price  60778 non-null  float64
 14  price             60778 non-null  float64
dtypes: float64(8), int64(4), object(3)
memory usage: 7.0+ MB


In [49]:
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
y = df['price']

numerical_cols = df.select_dtypes(include=['int64', "int32",'float64']).columns

# Drop 'carID' and 'price'
num_cols = [col for col in numerical_cols 
            if "_transformed" not in col and col not in ['price', 'carID']]

print(num_cols)
print(categorical_cols)

['mileage', 'tax', 'mpg', 'engineSize', 'previousOwners', 'stated_no_damage', 'age', 'mileage_per_year', 'efficiency_ratio', 'age_mileage', 'model_mean_price']
['Brand', 'transmission', 'fuelType']


In [50]:
#FILTER METHOD


#ANOVA FUNCTION

def anova_for_categorical(df, y, categorical_cols):
    # Align indices between df and y
    common_idx = df.index.intersection(y.index)
    df_aligned = df.loc[common_idx]
    y_aligned = y.loc[common_idx]
    
    f_scores, p_values = [], []
    for col in df_aligned.columns:
        if col in categorical_cols:
            groups = [y_aligned[df_aligned[col] == cat] for cat in df_aligned[col].dropna().unique()]
            if len(groups) > 1 and all(len(g) > 1 for g in groups):
                f_stat, p_val = stats.f_oneway(*groups)
            else:
                f_stat, p_val = 0.0, 1.0
        else:
            if df_aligned[col].nunique() > 1:
                # Remove NaN values for correlation calculation
                valid_idx = df_aligned[col].notna() & y_aligned.notna()
                if valid_idx.sum() > 1:
                    corr = np.corrcoef(df_aligned.loc[valid_idx, col], y_aligned[valid_idx])[0, 1]
                    f_stat = corr**2 * valid_idx.sum()
                    p_val = 0.0
                else:
                    f_stat, p_val = 0.0, 1.0
            else:
                f_stat, p_val = 0.0, 1.0
        f_scores.append(f_stat)
        p_values.append(p_val)
    
    return np.array(f_scores), np.array(p_values)


def filter_method_selection(X_train, y_train, categorical_cols, num_cols,
                            top_k=None,
                            var_threshold=0.01,
                            corr_threshold=0.85):
    
    print("FILTER METHOD (Variance + Spearman Correlation + ANOVA)")
    
    # Ensure indices match
    common_idx = X_train.index.intersection(y_train.index)
    X_train = X_train.loc[common_idx]
    y_train = y_train.loc[common_idx]
    
    # Filter numerical columns that exist in X_train
    num_cols_in_X = [col for col in num_cols if col in X_train.columns]
    
    # Variance Threshold
    if num_cols_in_X:
        vt_selector = VarianceThreshold(threshold=var_threshold)
        X_num_vt = pd.DataFrame(
            vt_selector.fit_transform(X_train[num_cols_in_X]),
            columns=np.array(num_cols_in_X)[vt_selector.get_support()],
            index=X_train.index
        )
        print(f"Removed {len(num_cols_in_X) - X_num_vt.shape[1]} low-variance numeric features.")
    else:
        X_num_vt = pd.DataFrame(index=X_train.index)
    
    # Spearman Correlation
    if not X_num_vt.empty and X_num_vt.shape[1] > 1:
        corr_matrix = X_num_vt.corr(method='spearman').abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        to_drop = [col for col in upper.columns if any(upper[col] > corr_threshold)]
        X_num_corr = X_num_vt.drop(columns=to_drop)
        print(f"Removed {len(to_drop)} correlated numeric features (Spearman |corr| > {corr_threshold}).")
    else:
        X_num_corr = X_num_vt
    
    # Filter categorical columns that exist in X_train
    categorical_cols_in_X = [col for col in categorical_cols if col in X_train.columns]
    
    # Combine numeric + categorical
    X_filtered = pd.concat([X_num_corr, X_train[categorical_cols_in_X]], axis=1)
    
    # ANOVA F-test
    f_scores, f_pvalues = anova_for_categorical(X_filtered, y_train, categorical_cols_in_X)
    f_norm = (f_scores - f_scores.min()) / (f_scores.max() - f_scores.min() + 1e-10)
    
    results_df = pd.DataFrame({
        'feature': X_filtered.columns,
        'ANOVA_F': f_scores,
        'ANOVA_p_value': f_pvalues,
        'ANOVA_norm': f_norm,
        'type': ['categorical' if c in categorical_cols_in_X else 'numerical' for c in X_filtered.columns]
    }).sort_values('ANOVA_norm', ascending=False)
    
    if top_k is None:
        selected_features = X_filtered.columns.tolist()
    else:
        selected_features = X_filtered.columns[np.argsort(f_norm)[-top_k:]].tolist()
    
    print(f"\nFilter method selected {len(selected_features)} features")
    
    return selected_features, X_filtered[selected_features], results_df



In [51]:
y = df["price"]

df = df.drop(columns=["price"])

In [52]:

#X_train = df[num_cols]
#y_train = df['price']

# Filter method (Variance + Spearman + ANOVA)
# Define categorical and numerical columns first (if not already defined)
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = df.select_dtypes(include=['int64',"int32", 'float64']).columns.tolist()

# Filter method with all required parameters
selected_filter, X_train_filtered, filter_df = filter_method_selection(
    df, 
    y,
    categorical_cols=categorical_cols,  # Add this
    num_cols=num_cols,                  # Add this
    top_k=None, 
    var_threshold=0.01, 
    corr_threshold=0.85
)

# Print features selected
print("\nFeatures selected by Filter Method:")
for f in selected_filter:
    print(f)

# Display feature importance scores
print("\nTop 10 Features by ANOVA Score:")
print(filter_df.head(10))



FILTER METHOD (Variance + Spearman Correlation + ANOVA)
Removed 0 low-variance numeric features.
Removed 2 correlated numeric features (Spearman |corr| > 0.85).

Filter method selected 12 features

Features selected by Filter Method:
mileage
tax
mpg
engineSize
previousOwners
stated_no_damage
age
efficiency_ratio
model_mean_price
Brand
transmission
fuelType

Top 10 Features by ANOVA Score:
             feature       ANOVA_F  ANOVA_p_value  ANOVA_norm         type
8   model_mean_price  34547.008697            0.0    1.000000    numerical
3         engineSize  22794.859457            0.0    0.659821    numerical
6                age  14620.501980            0.0    0.423206    numerical
10      transmission  13281.732054            0.0    0.384454  categorical
0            mileage  10577.242575            0.0    0.306169    numerical
1                tax   5798.312998            0.0    0.167838    numerical
2                mpg   5375.501179            0.0    0.155600    numerical
9       

In [53]:
#raise SystemExit("Stop before training the models")

## Test models on reduced DF

In [54]:
X_train_filter = X_train[["mileage", "tax", "mpg", "engineSize", "previousOwners", "stated_no_damage", "age", "efficiency_ratio", 
                          "Brand", "transmission", "model_mean_price", "fuelType"]]
X_train_filter

,mileage,tax,mpg,engineSize,previousOwners,stated_no_damage,age,efficiency_ratio,Brand,transmission,model_mean_price,fuelType
0,11899,126,51.4,1.4,2,1.0,4,34.266667,opel,manual,10503.556408,petrol
1,18500,145,48.9,1.4,4,1.0,2,32.600000,audi,semi-auto,22908.439241,petrol
2,10658,145,46.3,1.5,2,1.0,1,28.937500,mercedes,semi-auto,23606.673383,petrol
3,1500,145,43.5,0.0,4,1.0,0,435.000000,audi,manual,22661.237986,petrol
4,18495,100,64.0,2.1,4,1.0,3,29.090909,mercedes,automatic,23606.673383,diesel
...,...,...,...,...,...,...,...,...,...,...,...,...
60773,14867,150,55.4,1.4,3,1.0,2,36.933333,opel,manual,8321.809541,petrol
60774,20100,145,52.3,1.2,3,1.0,2,40.230769,toyota,manual,12412.360202,petrol
60775,10498,145,68.9,1.0,0,1.0,2,62.636364,toyota,manual,8070.361947,petrol
60776,17502,145,40.4,2.0,4,1.0,3,19.238095,audi,semi-auto,30342.439331,petrol


In [55]:
X_val_filter = X_val[["mileage", "tax", "mpg", "engineSize", "previousOwners", "stated_no_damage", "age", "efficiency_ratio", 
                          "Brand", "transmission", "model_mean_price", "fuelType"]]
X_val_filter

,mileage,tax,mpg,engineSize,previousOwners,stated_no_damage,age,efficiency_ratio,Brand,transmission,model_mean_price,fuelType
0,14774,145,53.3,2.0,0,1.0,1,25.380952,vw,manual,16879.259184,diesel
1,16375,150,47.9,1.6,4,1.0,4,28.176471,ford,manual,10246.530606,petrol
2,32746,265,37.2,3.0,3,0.0,4,12.000000,bmw,semi-auto,19699.821797,petrol
3,10164,145,68.9,1.5,3,1.0,1,43.062500,bmw,automatic,15827.569495,diesel
4,11407,145,80.7,1.5,2,1.0,1,50.437500,ford,manual,13395.112459,diesel
...,...,...,...,...,...,...,...,...,...,...,...,...
15190,10766,20,51.4,1.4,2,1.0,1,34.266667,opel,manual,10503.556408,petrol
15191,5000,145,48.7,2.0,1,1.0,0,23.190476,vw,semi-auto,16879.259184,diesel
15192,28056,145,60.1,2.0,0,1.0,3,28.619048,ford,manual,15778.868664,diesel
15193,25169,145,56.5,1.0,4,1.0,2,51.363636,ford,manual,10246.530606,petrol


### Linear Regression

In [56]:
LR = LinearRegression()
trainer = BrandModelTrainer(LR)

# fit
trainer.fit(X_train_filter, y_train)

# performance overall
trainer.evaluate_train(X_train_filter, y_train)
trainer.evaluate(X_val_filter, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train_filter, y_train)
trainer.evaluate_by_brand(X_val_filter, y_val)

Training models for 9 brands...

  ✓ opel done.
  ✓ audi done.
  ✓ mercedes done.
  ✓ ford done.
  ✓ bmw done.
  ✓ toyota done.
  ✓ hyundai done.
  ✓ vw done.
  ✓ skoda done.

Training Set Performance (Overall):
  RMSE: 3573.81
  MAE:  2222.15
  R²:   0.8613

Validation Set Performance (Overall):
  RMSE: 3519.67
  MAE:  2235.12
  R²:   0.8634

Training Performance per Brand:
   Brand     N        RMSE         MAE       R²
    opel  7640 1853.243137 1263.525248 0.730132
   skoda  3504 2185.729272 1507.377371 0.872988
    ford 13110 2221.794471 1588.261416 0.781843
  toyota  3776 2442.888906 1483.777615 0.844619
 hyundai  2724 2537.793064 1708.739414 0.823002
      vw  8479 3209.605225 2119.217054 0.826856
    audi  5971 4622.986455 2990.908133 0.843397
     bmw  6037 4847.060828 3308.061684 0.817756
mercedes  9537 5337.510529 3485.881963 0.745978

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1907 1950.044591 1321.161729 0.700290
   skoda  881

,Brand,N,RMSE,MAE,R²
7,opel,1907,1950.044591,1321.161729,0.700290
8,skoda,881,2249.534191,1530.980556,0.867319
5,hyundai,687,2282.681098,1611.756547,0.848298
1,ford,3278,2336.042981,1583.077034,0.759381
3,toyota,940,2813.057197,1597.891923,0.825037
0,vw,2125,3147.797050,2168.685325,0.835971
6,audi,1490,4415.497094,2951.564753,0.847531
2,bmw,1505,4801.691577,3303.615640,0.820433
4,mercedes,2382,5135.215991,3491.835771,0.758596


### Single tree

In [57]:
Tree = DecisionTreeRegressor(criterion="absolute_error")
trainer = BrandModelTrainer(Tree)

# fit
trainer.fit(X_train_filter, y_train)

# performance overall
trainer.evaluate_train(X_train_filter, y_train)
trainer.evaluate(X_val_filter, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train_filter, y_train)
trainer.evaluate_by_brand(X_val_filter, y_val)        # Performance VALID

Training models for 9 brands...

  ✓ opel done.
  ✓ audi done.
  ✓ mercedes done.
  ✓ ford done.
  ✓ bmw done.
  ✓ toyota done.
  ✓ hyundai done.
  ✓ vw done.
  ✓ skoda done.

Training Set Performance (Overall):
  RMSE: 104.41
  MAE:  5.77
  R²:   0.9999

Validation Set Performance (Overall):
  RMSE: 2854.93
  MAE:  1655.03
  R²:   0.9101

Training Performance per Brand:
   Brand     N       RMSE       MAE       R²
  toyota  3776  36.498308  1.571769 0.999965
    ford 13110  59.143592  3.164760 0.999845
      vw  8479  71.144767  4.987027 0.999915
   skoda  3504  81.232575  3.905251 0.999825
    audi  5971  91.272020  4.119578 0.999939
    opel  7640 115.867908 10.034293 0.998945
 hyundai  2724 122.123981  7.596549 0.999590
     bmw  6037 124.888708  3.789631 0.999879
mercedes  9537 164.661451 10.760931 0.999758

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1907 1606.384364 1041.118099 0.796619
    ford 3278 1897.636311 1187.793212 0.841221


,Brand,N,RMSE,MAE,R²
7,opel,1907,1606.384364,1041.118099,0.796619
1,ford,3278,1897.636311,1187.793212,0.841221
8,skoda,881,1916.737540,1297.177888,0.903673
5,hyundai,687,2034.726270,1226.984328,0.879465
3,toyota,940,2120.916044,1224.776553,0.900543
0,vw,2125,3130.114979,1702.693199,0.837808
6,audi,1490,3394.728552,2153.917612,0.909878
2,bmw,1505,3521.283126,2267.880902,0.903430
4,mercedes,2382,4106.184337,2473.310167,0.845651


In [58]:
KNR = KNeighborsRegressor()
trainer = BrandModelTrainer(KNR)

# fit
trainer.fit(X_train_filter, y_train)

# performance overall
trainer.evaluate_train(X_train_filter, y_train)
trainer.evaluate(X_val_filter, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train_filter, y_train)
trainer.evaluate_by_brand(X_val_filter, y_val)

Training models for 9 brands...

  ✓ opel done.
  ✓ audi done.
  ✓ mercedes done.
  ✓ ford done.
  ✓ bmw done.
  ✓ toyota done.
  ✓ hyundai done.
  ✓ vw done.
  ✓ skoda done.

Training Set Performance (Overall):
  RMSE: 2106.90
  MAE:  1279.35
  R²:   0.9518

Validation Set Performance (Overall):
  RMSE: 2653.18
  MAE:  1617.00
  R²:   0.9224

Training Performance per Brand:
   Brand     N        RMSE         MAE       R²
    opel  7640 1053.694252  709.095485 0.912760
    ford 13110 1251.262280  850.643022 0.930808
  toyota  3776 1295.274172  827.372689 0.956317
   skoda  3504 1560.093244 1116.381197 0.935293
 hyundai  2724 1620.480566  996.822500 0.927832
      vw  8479 1877.588841 1263.898586 0.940748
    audi  5971 2901.822295 1837.330792 0.938298
mercedes  9537 2924.337715 1879.424958 0.923748
     bmw  6037 3073.882105 1958.675710 0.926706

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1907 1445.251685  958.619872 0.835374
    ford 3278

,Brand,N,RMSE,MAE,R²
7,opel,1907,1445.251685,958.619872,0.835374
1,ford,3278,1624.221228,1047.505393,0.883679
3,toyota,940,1743.326649,1085.874875,0.932803
5,hyundai,687,1762.972849,1202.259160,0.909512
8,skoda,881,1786.394014,1321.542894,0.916329
0,vw,2125,2353.584548,1599.852623,0.908300
6,audi,1490,3448.410768,2288.695058,0.907005
2,bmw,1505,3746.992595,2422.754153,0.890654
4,mercedes,2382,3882.664379,2452.357828,0.861997


In [59]:
RF = RandomForestRegressor(random_state=69)
rf_trainer = BrandModelTrainer(RF)


rf_trainer.fit(X_train_filter, y_train)


rf_trainer.evaluate_train(X_train_filter, y_train)
rf_trainer.evaluate(X_val_filter, y_val)

rf_trainer.evaluate_train_by_brand(X_train_filter, y_train)
rf_trainer.evaluate_by_brand(X_val_filter, y_val)

Training models for 9 brands...

  ✓ opel done.
  ✓ audi done.
  ✓ mercedes done.
  ✓ ford done.
  ✓ bmw done.
  ✓ toyota done.
  ✓ hyundai done.
  ✓ vw done.
  ✓ skoda done.

Training Set Performance (Overall):
  RMSE: 788.13
  MAE:  469.51
  R²:   0.9933

Validation Set Performance (Overall):
  RMSE: 2027.59
  MAE:  1256.56
  R²:   0.9547

Training Performance per Brand:
   Brand     N        RMSE        MAE       R²
    opel  7640  467.382275 298.537130 0.982835
    ford 13110  508.887901 344.593448 0.988555
  toyota  3776  564.494504 337.391921 0.991703
 hyundai  2724  602.582997 367.218501 0.990021
   skoda  3504  631.736410 410.235779 0.989390
      vw  8479  706.981063 453.792085 0.991599
    audi  5971 1014.231488 627.762153 0.992462
     bmw  6037 1041.720655 638.596788 0.991582
mercedes  9537 1129.458861 689.349146 0.988625

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1907 1239.971469  818.720845 0.878819
 hyundai  687 1286.891712

,Brand,N,RMSE,MAE,R²
7,opel,1907,1239.971469,818.720845,0.878819
5,hyundai,687,1286.891712,884.574004,0.951785
3,toyota,940,1382.206957,916.673530,0.957759
1,ford,3278,1446.357182,922.507889,0.907760
8,skoda,881,1456.285159,1026.730184,0.944395
0,vw,2125,1867.099239,1222.317564,0.942291
6,audi,1490,2540.261094,1669.125358,0.949536
2,bmw,1505,2606.699005,1683.205149,0.947080
4,mercedes,2382,2942.758312,1896.157208,0.920725


### Neural Network

In [ ]:
from sklearn.neural_network import MLPRegressor

mlp_deep = MLPRegressor(
    hidden_layer_sizes=(256, 128, 64),  
    activation='relu',
    solver='adam',
    alpha=0.0001,  
    learning_rate_init=0.001,
    learning_rate='adaptive',
    max_iter=2000,
    batch_size=256,
    random_state=69,
    early_stopping=True,
    n_iter_no_change=30,  
    validation_fraction=0.15,
    verbose=True
)

mlp_trainer = BrandModelTrainer(mlp_deep)


mlp_trainer.fit(X_train_filter, y_train)


mlp_trainer.evaluate_train(X_train_filter, y_train)
mlp_trainer.evaluate(X_val_filter, y_val)


mlp_trainer.evaluate_train_by_brand(X_train_filter, y_train)
mlp_trainer.evaluate_by_brand(X_val_filter, y_val)

Training models for 9 brands...

Iteration 1, loss = 60223741.91105462
Validation score: -8.499781
Iteration 2, loss = 59305243.73648416
Validation score: -8.141860
Iteration 3, loss = 54311011.70361839
Validation score: -6.604598
Iteration 4, loss = 39077679.26747093
Validation score: -3.119478
Iteration 5, loss = 17349704.22868298
Validation score: -0.286623
Iteration 6, loss = 6842669.32004735
Validation score: 0.253421
Iteration 7, loss = 4574541.98373487
Validation score: 0.416557
Iteration 8, loss = 3643442.89648826
Validation score: 0.512893
Iteration 9, loss = 3102487.21743364
Validation score: 0.576590
Iteration 10, loss = 2738243.97477274
Validation score: 0.621080
Iteration 11, loss = 2430380.28119225
Validation score: 0.654328
Iteration 12, loss = 2228989.22444136
Validation score: 0.679126
Iteration 13, loss = 2072881.88185177
Validation score: 0.695507
Iteration 14, loss = 1960234.96096304
Validation score: 0.708916
Iteration 15, loss = 1877569.67398978
Validation score: 

c:\Users\morit\anaconda3\envs\DataMining\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:788: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


  ✓ vw done.
Iteration 1, loss = 120998905.52754487
Validation score: -5.229427
Iteration 2, loss = 120880361.83692554
Validation score: -5.218358
Iteration 3, loss = 120517438.54874848
Validation score: -5.186479
Iteration 4, loss = 119550014.55574410
Validation score: -5.106757
Iteration 5, loss = 117285603.36957321
Validation score: -4.933171
Iteration 6, loss = 112701052.97455740
Validation score: -4.601372
Iteration 7, loss = 104499460.94121246
Validation score: -4.041990
Iteration 8, loss = 91414734.78050460
Validation score: -3.222218
Iteration 9, loss = 73513067.36346826
Validation score: -2.205951
Iteration 10, loss = 53201145.03586544
Validation score: -1.178995
Iteration 11, loss = 34034232.89337071
Validation score: -0.311240


### Results

Performance on the full dataset
| *Model* | *RMSE* | *MAE* | *R²* |
|---------|--------|--------|------|
| Linear Regression | 3383.27 | 2156.67 | 0.8723 |
| Tree | 2784.00 | 1663.58 | 0.9135 |
| KNR|2623.89 |1599.60 |0.9232 |
|RF | 2032.89 |1258.34 |0.9539 |
|NN | 2319.24 |1466.96 |0.9400 |



Performance on the reduced dataset
| *Model* | *RMSE* | *MAE* | *R²* |
|---------|--------|--------|------|
| Linear Regression | 3501.39 | 2241.68 | 0.8632 |
| Tree       | 2759.39 | 1650.79 | 0.9150 |
| KNR               | 2624.75 | 1598.18 | 0.9231 |
| RF                | **2036.38** | **1254.22** | **0.9537** |
| NN                | 2337.17.41 | 1476.69 | 0.9391 |



In [ ]:
#raise SystemExit("Stop before training the models")

SystemExit: Stop before training the models

C:\Users\liber\AppData\Roaming\Python\Python311\site-packages\IPython\core\interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## Feature Selection - Wrapper Method

In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

print(num_cols)

['mileage', 'tax', 'mpg', 'engineSize', 'previousOwners', 'stated_no_damage', 'age', 'mileage_per_year', 'efficiency_ratio', 'age_mileage', 'model_mean_price']


In [ ]:
def rfe(X_train, X_val, y_train, y_val, num_cols, step=1, n_estimators=100, random_state=69):
    
    valid_num_cols = [col for col in num_cols if col in X_train.columns]
    if len(valid_num_cols) == 0:
        raise ValueError("No valid numeric columns found in X_train.")
    print(f"Using {len(valid_num_cols)} numeric columns for RFE:\n{valid_num_cols}")
    
    X_train_num = X_train[valid_num_cols]
    X_val_num = X_val[valid_num_cols]
    nof_list = np.arange(1, X_train_num.shape[1]+1)
    
    best_mae = float('inf')
    nof = 0
    train_mae_list = []
    val_mae_list = []
    features_to_select = None
    
    for n in nof_list:
        model = RandomForestRegressor(n_estimators=n_estimators, random_state=random_state, n_jobs=-1)
        rfe = RFE(estimator=model, n_features_to_select=n, step=step)
        X_train_rfe = rfe.fit_transform(X_train_num, y_train)
        X_val_rfe = rfe.transform(X_val_num)
        
        model.fit(X_train_rfe, y_train)
        
        train_pred = model.predict(X_train_rfe)
        val_pred = model.predict(X_val_rfe)
        train_mae = mean_absolute_error(y_train, train_pred)
        val_mae = mean_absolute_error(y_val, val_pred)
        
        train_mae_list.append(train_mae)
        val_mae_list.append(val_mae)
        
        if val_mae < best_mae:
            best_mae = val_mae
            nof = n
            features_to_select = pd.Series(rfe.support_, index=X_train_num.columns)
    
    selected_features = features_to_select[features_to_select].index.tolist()
    
    print(f"\nOptimum number of features: {nof}")
    print(f"Best validation MAE: {best_mae:.4f}")
    print("Selected features:")
    print(selected_features)
    
    return nof, best_mae, selected_features, train_mae_list, val_mae_list

In [ ]:
nof, best_score, selected_rfe_features, train_scores, val_scores = rfe(
    X_train, X_val, y_train, y_val, num_cols
)

Using 11 numeric columns for RFE:
['mileage', 'tax', 'mpg', 'engineSize', 'previousOwners', 'stated_no_damage', 'age', 'mileage_per_year', 'efficiency_ratio', 'age_mileage', 'model_mean_price']

Optimum number of features: 9
Best validation MAE: 1323.6319
Selected features:
['mileage', 'tax', 'mpg', 'engineSize', 'age', 'mileage_per_year', 'efficiency_ratio', 'age_mileage', 'model_mean_price']


In [ ]:
X_train_rfed = X_train[["mileage", "tax", "mpg", "engineSize", "age", "efficiency_ratio", "mileage_per_year", "age_mileage",
                          "Brand", "transmission", "model_mean_price", "fuelType"]]
X_train_rfed

,mileage,tax,mpg,engineSize,age,efficiency_ratio,mileage_per_year,age_mileage,Brand,transmission,model_mean_price,fuelType
0,16326,125,51.4,1.4,3,34.266667,4081.500000,0.48978,opel,manual,10496.007929,petrol
1,12510,145,56.5,1.0,2,51.363636,4170.000000,0.25020,toyota,manual,8029.315503,petrol
2,6999,144,58.2,1.0,1,52.909091,3499.500000,0.06999,ford,manual,13441.487966,petrol
3,28465,160,43.5,2.0,3,20.714286,7116.250000,0.85395,mercedes,manual,30801.319510,petrol
4,9221,145,33.6,2.0,1,16.000000,4610.500000,0.09221,vw,semi-auto,34233.178082,electric
...,...,...,...,...,...,...,...,...,...,...,...,...
60773,21131,145,67.3,2.1,1,30.590909,10565.500000,0.21131,mercedes,semi-auto,20327.302521,diesel
60774,4722,145,50.4,1.6,1,29.647059,2361.000000,0.04722,vw,manual,22619.961912,diesel
60775,24755,0,76.3,1.6,5,44.882353,4125.833333,1.23775,audi,manual,14343.507937,diesel
60776,42000,125,57.6,2.0,5,27.428571,7000.000000,2.10000,bmw,manual,15717.953402,diesel


In [ ]:
X_val_rfed = X_val[["mileage", "tax", "mpg", "engineSize", "age", "efficiency_ratio", "mileage_per_year", "age_mileage",
                          "Brand", "transmission", "model_mean_price", "fuelType"]]
X_val_rfed

,mileage,tax,mpg,engineSize,age,efficiency_ratio,mileage_per_year,age_mileage,Brand,transmission,model_mean_price,fuelType
0,5000,145,48.7,2.0,1,23.190476,2500.000000,0.05000,vw,automatic,17101.053465,diesel
1,10,145,43.5,1.4,1,29.000000,5.000000,0.00010,opel,manual,8348.382147,petrol
2,46000,125,51.4,1.0,4,46.727273,9200.000000,1.84000,ford,semi-auto,13441.487966,petrol
3,24219,20,60.1,1.2,3,46.230769,6054.750000,0.72657,vw,manual,11435.947368,petrol
4,20251,145,46.3,1.8,4,24.368421,4050.200000,0.81004,toyota,automatic,10373.418919,petrol
...,...,...,...,...,...,...,...,...,...,...,...,...
15190,33154,30,54.3,1.2,6,41.769231,4736.285714,1.98924,ford,manual,10245.037681,petrol
15191,19000,20,60.1,1.0,4,54.636364,3800.000000,0.76000,vw,manual,11435.947368,petrol
15192,3971,145,45.6,1.5,1,28.500000,1985.500000,0.03971,vw,manual,16722.731829,petrol
15193,56603,0,83.1,1.4,4,55.400000,11320.600000,2.26412,vw,manual,11435.947368,diesel


In [ ]:
RF = RandomForestRegressor(random_state=69)
rf_trainer = BrandModelTrainer(RF)


rf_trainer.fit(X_train_rfed, y_train)


rf_trainer.evaluate_train(X_train_rfed, y_train)
rf_trainer.evaluate(X_val_rfed, y_val)

rf_trainer.evaluate_train_by_brand(X_train_rfed, y_train)
rf_trainer.evaluate_by_brand(X_val_rfed, y_val)

Training models for 9 brands...

  ✓ opel done.
  ✓ toyota done.
  ✓ ford done.
  ✓ mercedes done.
  ✓ vw done.
  ✓ bmw done.
  ✓ audi done.
  ✓ hyundai done.
  ✓ skoda done.

Training Set Performance (Overall):
  RMSE: 791.34
  MAE:  473.66
  R²:   0.9932

Validation Set Performance (Overall):
  RMSE: 2033.05
  MAE:  1256.58
  R²:   0.9539

Training Performance per Brand:
   Brand     N        RMSE        MAE       R²
    opel  7644  463.914285 309.235998 0.982848
    ford 13108  535.870395 345.862177 0.987429
  toyota  3775  561.716833 338.077681 0.992100
 hyundai  2724  577.270768 367.348817 0.990626
   skoda  3507  602.826087 409.353137 0.990329
      vw  8479  708.249640 454.007482 0.991500
    audi  5971 1037.789691 645.645367 0.992224
     bmw  6029 1039.159002 641.595021 0.991591
mercedes  9541 1125.371000 692.305729 0.988845

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1906 1151.895521  786.584062 0.895455
    ford 3282 1354.041743

,Brand,N,RMSE,MAE,R²
1,opel,1906,1151.895521,786.584062,0.895455
2,ford,3282,1354.041743,935.275800,0.918098
5,hyundai,683,1438.914752,944.670402,0.943047
7,skoda,875,1554.184116,1073.572999,0.936224
3,toyota,941,1560.618769,952.116895,0.937747
0,vw,2125,1904.666983,1228.593323,0.941829
6,audi,1489,2602.379454,1662.530539,0.943410
8,bmw,1513,2680.604426,1713.106259,0.944944
4,mercedes,2381,2881.876592,1833.798991,0.920858


Final Results

Performance on the full dataset
| *Model* | *RMSE* | *MAE* | *R²* |
|---------|--------|--------|------|
| Linear Regression | 3383.27 | 2156.67 | 0.8723 |
| Tree | 2784.00 | 1663.58 | 0.9135 |
| KNR|2623.89 |1599.60 |0.9232 |
|RF | 2032.89 |1258.34 |0.9539 |
|NN | 2319.24 |1466.96 |0.9400 |



Performance on the reduced dataset
| *Model* | *RMSE* | *MAE* | *R²* |
|---------|--------|--------|------|
| Linear Regression | 3501.39 | 2241.68 | 0.8632 |
| Tree       | 2759.39 | 1650.79 | 0.9150 |
| KNR               | 2624.75 | 1598.18 | 0.9231 |
| RF                | **2036.38** | **1254.22** | **0.9537** |
| NN                | 2337.17.41 | 1476.69 | 0.9391 |


Random Forest on the rfed dataset

|*Model*| *RMSE*| *MAE* | *R²*|
|----|----|----|----|
|RF| 2033.05| 1256.58 | 0.9539|

## Hyperparam tuning

In [ ]:
class HoldoutRandomSearch:
    def __init__(self, trainer_class, param_space, n_iter=20, optimize_metric='mae'):
        self.trainer_class = trainer_class
        self.param_space = param_space
        self.n_iter = n_iter
        self.optimize_metric = optimize_metric.lower()
        self.results = []
        self.best_params = None
        self.best_score = None
        self.best_trainer = None
        self.brand_best_configs = {}
        
    def sample_params(self):
        if isinstance(self.param_space, list):
            return random.choice(self.param_space)
        else:
            return {k: random.choice(v) for k, v in self.param_space.items()}
    
    def _get_param_summary(self, params):
        summary = {}
        
        if 'n_estimators' in params:
            summary['n_estimators'] = params['n_estimators']
        if 'max_depth' in params:
            summary['max_depth'] = params['max_depth']
        if 'max_features' in params:
            summary['max_features'] = params['max_features']
            
        if 'hidden_layer_sizes' in params:
            summary['hidden_layers'] = str(params['hidden_layer_sizes'])
        if 'alpha' in params:
            summary['alpha'] = params['alpha']
        if 'learning_rate_init' in params:
            summary['lr'] = params['learning_rate_init']
        if 'activation' in params:
            summary['activation'] = params['activation']
            
        return summary
    
    def run(self, X_train, y_train, X_val, y_val):
        metric_name = self.optimize_metric.upper()
        print(f"Running random search ({self.n_iter} iterations)...")
        print(f"Optimizing for: {metric_name}\n")
        
        start_time = time.time()
        brands = X_train["Brand"].unique()

        for brand in brands:
            self.brand_best_configs[brand] = {
                'score': float('inf'),
                'params': None,
                'config_num': None,
                'all_metrics': {}
            }
        
        for i in range(self.n_iter):
            iter_start = time.time()
            params = self.sample_params()
            
            print(f"\n{'='*70}")
            print(f"[{i+1}/{self.n_iter}] Testing params:")
            print(params)
            print('='*70)
            

            estimator = self.trainer_class.estimator.__class__(**params)
            trainer = BrandModelTrainer(estimator)
            trainer.fit(X_train, y_train)

            metrics = trainer.evaluate(X_val, y_val)
            rmse = metrics["RMSE"]
            mae = metrics["MAE"]
            r2 = metrics["R²"]
            
            current_score = mae if self.optimize_metric == 'mae' else rmse
            
            brand_results = {}
            for brand in brands:
                mask = X_val["Brand"] == brand
                y_true_brand = y_val[mask]
                y_pred_brand = trainer.predict(X_val[mask])
                
                brand_rmse = np.sqrt(mean_squared_error(y_true_brand, y_pred_brand))
                brand_mae = mean_absolute_error(y_true_brand, y_pred_brand)
                brand_r2 = r2_score(y_true_brand, y_pred_brand)
                
                brand_score = brand_mae if self.optimize_metric == 'mae' else brand_rmse
                
                brand_results[brand] = {
                    'score': brand_score,
                    'rmse': brand_rmse,
                    'mae': brand_mae,
                    'r2': brand_r2
                }
                
                if brand_score < self.brand_best_configs[brand]['score']:
                    self.brand_best_configs[brand]['score'] = brand_score
                    self.brand_best_configs[brand]['params'] = params.copy()
                    self.brand_best_configs[brand]['config_num'] = i + 1
                    self.brand_best_configs[brand]['all_metrics'] = {
                        'rmse': brand_rmse,
                        'mae': brand_mae,
                        'r2': brand_r2
                    }
            
            self.results.append({
                "config_num": i + 1,
                "params": params,
                "rmse": rmse,
                "mae": mae,
                "r2": r2,
                "score": current_score,
                "brand_results": brand_results
            })

            if self.best_score is None or current_score < self.best_score:
                self.best_score = current_score
                self.best_params = params
                self.best_trainer = trainer
                print(f"New best overall model ({metric_name}: {current_score:.2f})")
           

            elapsed_total = time.time() - start_time
            avg_per_iter = elapsed_total / (i + 1)
            eta = avg_per_iter * (self.n_iter - (i + 1))
            
            print(f"\nProgress: {i+1}/{self.n_iter} | Elapsed: {elapsed_total:.1f}s | ETA: ~{eta:.1f}s")
        
        print(f"SEARCH COMPLETED - FINAL RESULTS (Optimized for {metric_name})")
        
        print(f"\n Best general model:")
        print(f"  Best {metric_name}: {self.best_score:.2f}")
        
        best_result = [r for r in self.results if r['score'] == self.best_score][0]
        print(f"  RMSE: {best_result['rmse']:.2f}")
        print(f"  MAE:  {best_result['mae']:.2f}")
        print(f"  R²:   {best_result['r2']:.4f}")
        print(f"  Params: {self.best_params}")
   
        print(f"Best configuration per brand (by {metric_name})")

        
        brand_summary = []
        for brand in sorted(brands):
            config = self.brand_best_configs[brand]
            
            summary = {
                'Brand': brand,
                f'Best_{metric_name}': config['score'],
                'RMSE': config['all_metrics']['rmse'],
                'MAE': config['all_metrics']['mae'],
                'R²': config['all_metrics']['r2'],
                'Config_Num': config['config_num']
            }
            
            param_summary = self._get_param_summary(config['params'])
            summary.update(param_summary)
            
            brand_summary.append(summary)
            
            print(f"\n{brand.upper()}:")
            print(f"  Best {metric_name}: {config['score']:.2f}")
            print(f"  RMSE: {config['all_metrics']['rmse']:.2f}")
            print(f"  MAE:  {config['all_metrics']['mae']:.2f}")
            print(f"  R²:   {config['all_metrics']['r2']:.4f}")
            print(f"  Found at iteration: {config['config_num']}")
            print(f"  Best params:")
            for k, v in list(config['params'].items())[:5]:
                if k not in ['random_state', 'n_jobs', 'shuffle', 'verbose', 'warm_start']:
                    print(f"    {k}: {v}")
        
        df_summary = pd.DataFrame(brand_summary).sort_values(f'Best_{metric_name}')
  
        print(df_summary.to_string(index=False))
        
        results_df = pd.DataFrame([
            {
                'Config': r['config_num'],
                metric_name: r['score'],
                'RMSE': r['rmse'],
                'MAE': r['mae'],
                'R²': r['r2']
            }
            for r in self.results
        ]).sort_values(metric_name)
        
        print(f"ALL CONFIGURATIONS (sorted by {metric_name}):")
        print(results_df.head(10).to_string(index=False))
        
        return self.best_trainer, self.best_params, self.best_score

### Random Forest

In [ ]:
base_estimator = RandomForestRegressor(random_state=69, n_jobs=-1)
trainer = BrandModelTrainer(base_estimator)

param_space_rf = {
    "n_estimators": [400, 800, 1200],
    "max_depth": [None, 15, 25, 35],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": [0.3, 0.5, 0.7, 1.0],
    "min_impurity_decrease": [0.0, 1e-4, 1e-3, 1e-2],
    "max_samples": [0.5, 0.7, 0.9, None],
    "criterion": ["squared_error", "friedman_mse"],
    "bootstrap": [True],
    "random_state": [69],
    "n_jobs": [-1]
}

search = HoldoutRandomSearch(
    trainer_class=trainer,
    param_space=param_space_rf,
    n_iter=60
)


best_trainer, best_params, best_rmse = search.run(X_train_filter, y_train, X_val_filter, y_val)


Running random search (60 iterations)...
Optimizing for: MAE


[1/60] Testing params:
{'n_estimators': 1200, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 0.3, 'min_impurity_decrease': 0.0001, 'max_samples': 0.9, 'criterion': 'friedman_mse', 'bootstrap': True, 'random_state': 69, 'n_jobs': -1}
Training models for 9 brands...

  ✓ opel done.
  ✓ toyota done.
  ✓ ford done.
  ✓ mercedes done.
  ✓ vw done.
  ✓ bmw done.
  ✓ audi done.
  ✓ hyundai done.
  ✓ skoda done.

Validation Set Performance (Overall):
  RMSE: 2010.32
  MAE:  1245.43
  R²:   0.9549
New best overall model (MAE: 1245.43)

Progress: 1/60 | Elapsed: 84.7s | ETA: ~4996.7s

[2/60] Testing params:
{'n_estimators': 1200, 'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 1.0, 'min_impurity_decrease': 0.001, 'max_samples': 0.5, 'criterion': 'squared_error', 'bootstrap': True, 'random_state': 69, 'n_jobs': -1}
Training models for 9 brands...

  ✓ opel done.
  ✓ to

In [ ]:
best_trainer.evaluate_by_brand(X_val_filter, y_val)



Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1906 1086.769229  748.110206 0.906943
    ford 3282 1277.274084  901.254733 0.927122
 hyundai  683 1368.252148  890.840175 0.948504
  toyota  941 1378.231055  858.386788 0.951447
   skoda  875 1494.197535 1037.671806 0.941052
      vw 2125 1875.588743 1188.962644 0.943592
     bmw 1513 2457.630204 1624.097258 0.953722
    audi 1489 2543.390249 1621.939526 0.945946
mercedes 2381 2793.058290 1790.165923 0.925661


,Brand,N,RMSE,MAE,R²
1,opel,1906,1086.769229,748.110206,0.906943
2,ford,3282,1277.274084,901.254733,0.927122
5,hyundai,683,1368.252148,890.840175,0.948504
3,toyota,941,1378.231055,858.386788,0.951447
7,skoda,875,1494.197535,1037.671806,0.941052
0,vw,2125,1875.588743,1188.962644,0.943592
8,bmw,1513,2457.630204,1624.097258,0.953722
6,audi,1489,2543.390249,1621.939526,0.945946
4,mercedes,2381,2793.058290,1790.165923,0.925661


In [ ]:
#raise SystemExit("Stop before training the models")

## Predictions

In [ ]:
best_params_per_brand = {
    'opel': {
        'n_estimators': 1200,
        'max_depth': 15,
        'min_samples_split': 2,
        'min_samples_leaf': 1,
        'max_features': 0.5,
        'min_impurity_decrease': 0.0,
        'max_samples': 0.7,
        'criterion': 'squared_error',
        'bootstrap': True,
        'random_state': 69,
        'n_jobs': -1
    },
    'toyota': {
        'n_estimators': 800,
        'max_depth': 25,
        'min_samples_split': 5,
        'min_samples_leaf': 1,
        'max_features': 0.3,
        'min_impurity_decrease': 0.01,
        'max_samples': None,
        'criterion': 'friedman_mse',
        'bootstrap': True,
        'random_state': 69,
        'n_jobs': -1
    },
    'ford': {
        'n_estimators': 1200,
        'max_depth': 15,
        'min_samples_split': 2,
        'min_samples_leaf': 1,
        'max_features': 0.5,
        'min_impurity_decrease': 0.0,
        'max_samples': 0.7,
        'criterion': 'squared_error',
        'bootstrap': True,
        'random_state': 69,
        'n_jobs': -1
    },
    'hyundai': {
        'n_estimators': 1200,
        'max_depth': None,
        'min_samples_split': 5,
        'min_samples_leaf': 1,
        'max_features': 0.5,
        'min_impurity_decrease': 0.0,
        'max_samples': None,
        'criterion': 'squared_error',
        'bootstrap': True,
        'random_state': 69,
        'n_jobs': -1
    },
    'skoda': {
        'n_estimators': 1200,
        'max_depth': 15,
        'min_samples_split': 2,
        'min_samples_leaf': 1,
        'max_features': 0.5,
        'min_impurity_decrease': 0.0,
        'max_samples': 0.7,
        'criterion': 'squared_error',
        'bootstrap': True,
        'random_state': 69,
        'n_jobs': -1
    },
    'vw': {
        'n_estimators': 1200,
        'max_depth': None,
        'min_samples_split': 5,
        'min_samples_leaf': 1,
        'max_features': 0.5,
        'min_impurity_decrease': 0.0,
        'max_samples': None,
        'criterion': 'squared_error',
        'bootstrap': True,
        'random_state': 69,
        'n_jobs': -1
    },
    'bmw': {
        'n_estimators': 400,
        'max_depth': 35,
        'min_samples_split': 2,
        'min_samples_leaf': 1,
        'max_features': 0.5,
        'min_impurity_decrease': 0.0,
        'max_samples': None,
        'criterion': 'squared_error',
        'bootstrap': True,
        'random_state': 69,
        'n_jobs': -1
    },
    'audi': {
        'n_estimators': 1200,
        'max_depth': None,
        'min_samples_split': 5,
        'min_samples_leaf': 1,
        'max_features': 0.5,
        'min_impurity_decrease': 0.0,
        'max_samples': None,
        'criterion': 'squared_error',
        'bootstrap': True,
        'random_state': 69,
        'n_jobs': -1
    },
    'mercedes': {
        'n_estimators': 1200,
        'max_depth': None,
        'min_samples_split': 5,
        'min_samples_leaf': 1,
        'max_features': 0.5,
        'min_impurity_decrease': 0.0,
        'max_samples': None,
        'criterion': 'squared_error',
        'bootstrap': True,
        'random_state': 69,
        'n_jobs': -1
    }
}

In [ ]:
X_full_train = pd.concat([X_train_filter, X_val_filter], axis=0)
y_full_train = pd.concat([y_train, y_val], axis=0)

numeric_cols = ['mileage', 'tax', 'mpg', 'engineSize', 'age', 
                 'efficiency_ratio', 
                'model_mean_price']

categorical_cols = ['Brand', 'transmission', 'fuelType']


In [ ]:
models = {}
scalers = {}
brand_encoders = {}


brands = X_full_train['Brand'].unique()

for brand in brands:
    brand_lower = brand.lower()
    print(f"\n Training model for {brand.upper()}")
    
    mask_train = X_full_train['Brand'] == brand
    X_brand_train = X_full_train[mask_train].copy()
    y_brand_train = y_full_train[mask_train].copy()
    
    X_numeric = X_brand_train[numeric_cols].copy()
    
    scaler = RobustScaler()
    X_numeric_scaled = scaler.fit_transform(X_numeric)
    X_numeric_scaled = pd.DataFrame(
        X_numeric_scaled, 
        columns=numeric_cols,
        index=X_brand_train.index
    )

    scalers[brand] = scaler

    X_brand_encoded = X_numeric_scaled.copy()
    
    for col in ['transmission', 'fuelType']:
        dummies = pd.get_dummies(X_brand_train[col], prefix=col, drop_first=True)
        X_brand_encoded = pd.concat([X_brand_encoded, dummies], axis=1)
    

    brand_encoders[brand] = {
        'columns': X_brand_encoded.columns.tolist(),
        'numeric_cols': numeric_cols
    }
    
    if brand_lower in best_params_per_brand:
        params = best_params_per_brand[brand_lower]
    else:
        params = {
            'n_estimators': 1200,
            'max_depth': None,
            'min_samples_split': 5,
            'min_samples_leaf': 1,
            'max_features': 0.5,
            'random_state': 69,
            'n_jobs': -1
        }
    
    model = RandomForestRegressor(**params)
    model.fit(X_brand_encoded, y_brand_train)
  
    models[brand] = model
    
    print(f" {brand.upper()} model trained")


 Training model for OPEL
 OPEL model trained

 Training model for AUDI
 AUDI model trained

 Training model for MERCEDES
 MERCEDES model trained

 Training model for FORD
 FORD model trained

 Training model for BMW
 BMW model trained

 Training model for TOYOTA
 TOYOTA model trained

 Training model for HYUNDAI
 HYUNDAI model trained

 Training model for VW
 VW model trained

 Training model for SKODA
 SKODA model trained


In [ ]:
test_processed.head()

,year,mileage,tax,mpg,engineSize,previousOwners,stated_no_damage,Brand,transmission,model,fuelType
carID,,,,,,,,,,,
89856,2015,30700,205,41.5,1.6,3,1.0,hyundai,automatic,i30,petrol
106581,2017,48191,150,38.2,2.0,2,1.0,vw,semi-auto,tiguan,petrol
80886,2016,36792,125,51.4,1.5,2,1.0,bmw,automatic,2 series,petrol
100174,2019,5533,145,44.1,1.2,1,1.0,opel,manual,grandland x,petrol
81376,2019,9058,150,51.4,2.0,4,1.0,bmw,semi-auto,1 series,diesel


In [ ]:
X_test = minimal_features(test_processed)
X_test

,mileage,tax,mpg,engineSize,previousOwners,stated_no_damage,Brand,transmission,model,fuelType,age,mileage_per_year,efficiency_ratio,age_mileage
carID,,,,,,,,,,,,,,
89856,30700,205,41.5,1.6,3,1.0,hyundai,automatic,i30,petrol,5,5116.666667,24.411765,1.53500
106581,48191,150,38.2,2.0,2,1.0,vw,semi-auto,tiguan,petrol,3,12047.750000,18.190476,1.44573
80886,36792,125,51.4,1.5,2,1.0,bmw,automatic,2 series,petrol,4,7358.400000,32.125000,1.47168
100174,5533,145,44.1,1.2,1,1.0,opel,manual,grandland x,petrol,1,2766.500000,33.923077,0.05533
81376,9058,150,51.4,2.0,4,1.0,bmw,semi-auto,1 series,diesel,1,4529.000000,24.476190,0.09058
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105775,27575,145,46.3,1.4,1,1.0,vw,manual,tiguan,petrol,3,6893.750000,30.866667,0.82725
81363,1980,145,34.0,2.0,3,1.0,bmw,automatic,x2,petrol,0,1980.000000,16.190476,0.00000
76833,8297,145,38.2,2.0,4,1.0,audi,semi-auto,q5,diesel,1,4148.500000,18.190476,0.08297


In [ ]:
X_test['model_mean_price'] = X_test['model'].map(model_mean_price)

In [ ]:
X_train_filter.head()

,mileage,tax,mpg,engineSize,previousOwners,stated_no_damage,age,efficiency_ratio,Brand,transmission,model_mean_price,fuelType
0,11899,124,51.4,1.4,2,1.0,4,34.266667,opel,manual,10488.218094,petrol
1,18500,145,48.7,1.4,4,1.0,2,32.466667,audi,semi-auto,22860.502532,petrol
2,10658,145,46.3,1.5,2,1.0,1,28.937500,mercedes,semi-auto,23609.185947,petrol
3,1500,145,43.5,0.0,4,1.0,0,435.000000,audi,manual,22522.319540,petrol
4,18495,98,64.1,2.1,4,1.0,3,29.136364,mercedes,automatic,23609.185947,diesel


In [ ]:
X_test = X_test.drop(columns=["model", "mileage_per_year", "age_mileage"])
X_test.head()


,mileage,tax,mpg,engineSize,previousOwners,stated_no_damage,Brand,transmission,fuelType,age,efficiency_ratio,model_mean_price
carID,,,,,,,,,,,,
89856,30700,205,41.5,1.6,3,1.0,hyundai,automatic,petrol,5,24.411765,11848.081505
106581,48191,150,38.2,2.0,2,1.0,vw,semi-auto,petrol,3,18.190476,21509.106996
80886,36792,125,51.4,1.5,2,1.0,bmw,automatic,petrol,4,32.125000,19665.944282
100174,5533,145,44.1,1.2,1,1.0,opel,manual,petrol,1,33.923077,17217.228906
81376,9058,150,51.4,2.0,4,1.0,bmw,semi-auto,diesel,1,24.476190,15836.032550


In [ ]:
predictions = np.zeros(len(X_test))

for brand in X_test['Brand'].unique():
    print(f"\n Predicting for {brand.upper()}...")

    mask_test = X_test['Brand'] == brand
    X_brand_test = X_test[mask_test].copy()
    
    X_numeric_test = X_brand_test[numeric_cols].copy()
    
    scaler = scalers[brand]
    X_numeric_test_scaled = scaler.transform(X_numeric_test)
    X_numeric_test_scaled = pd.DataFrame(
        X_numeric_test_scaled,
        columns=numeric_cols,
        index=X_brand_test.index
    )
    
    X_brand_test_encoded = X_numeric_test_scaled.copy()
    
    for col in ['transmission', 'fuelType']:
        dummies = pd.get_dummies(X_brand_test[col], prefix=col, drop_first=True)
        X_brand_test_encoded = pd.concat([X_brand_test_encoded, dummies], axis=1)

    expected_cols = brand_encoders[brand]['columns']

    for col in expected_cols:
        if col not in X_brand_test_encoded.columns:
            X_brand_test_encoded[col] = 0
    
    X_brand_test_encoded = X_brand_test_encoded[expected_cols]
    
    model = models[brand]
    brand_predictions = model.predict(X_brand_test_encoded)

    predictions[mask_test] = brand_predictions
    



 Predicting for HYUNDAI...

 Predicting for VW...

 Predicting for BMW...

 Predicting for OPEL...

 Predicting for FORD...

 Predicting for MERCEDES...

 Predicting for SKODA...

 Predicting for TOYOTA...

 Predicting for AUDI...


In [ ]:
if X_test.index.name == 'carID' or 'carID' in X_test.index.names:
    car_ids = X_test.index
else:
    car_ids = X_test.index

submission = pd.DataFrame({
    'carID': car_ids,
    'price': predictions
})

submission = submission.sort_values('carID')


submission['price'] = submission['price'].round(2)

output_filename = 'submission.csv'
submission.to_csv(output_filename, index=False)
